# NEXLYTICS — Predictive Analytics & Machine Learning Pipeline

Welcome to the **NEXLYTICS ML Analysis & Deliverables Notebook**.

This notebook integrates the core modules developed across the NEXLYTICS ML engine:
1. **Environment Setup & Imports** (`src` modules)
2. **Data Ingestion & Exploratory Data Analysis** (`src.data_processing`)
3. **Trend Visualization** (`src.evaluation.plot_trend_chart`)
4. **Anomaly Detection & Outlier Attribution** (`src.anomaly_detection`)
5. **Predictive Sales Forecasting** (`src.forecasting`)
6. **Model Evaluation & Naive Benchmarks** (`src.evaluation`)
7. **Artifacts & Deliverables Export** (CSVs, figures, and markdown reports)

## 1. Environment Setup & Module Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import modular NEXLYTICS ML packages
import src.data_processing as dp
import src.anomaly_detection as ad
import src.forecasting as fc
import src.evaluation as ev

print("All NEXLYTICS ML modules loaded successfully.")

## 2. Data Ingestion & Exploratory Analysis (EDA)

Loads raw transaction logs, validates the schema (`transaction_date`, `transaction_id`, `revenue`, `cost`, `profit`, `category`), cleans data safely, and aggregates daily metrics.

In [ ]:
# Specify path to transaction CSV
DATA_PATH = Path("../data/transactions.csv")

if DATA_PATH.exists():
    raw_df = dp.load_dataset(DATA_PATH)
    cleaned_txns, data_summary = dp.validate_and_clean_transactions(raw_df)
    daily_df = dp.aggregate_daily_transactions(cleaned_txns)
    missing_dates = dp.detect_missing_dates(daily_df)
    
    print(f"Loaded {data_summary['total_records']} transactions.")
    print(f"Daily aggregated days: {len(daily_df)}. Missing calendar dates: {len(missing_dates)}")
    display(daily_df.head())
else:
    print(f"Notice: No local data file found at {DATA_PATH}.")
    print("Place your real transaction CSV at data/transactions.csv or run via CLI: python run_pipeline.py --input <path>")

## 3. Financial Trend Visualization

Generates publication-ready daily trends for Revenue, Cost, and Profit.

In [ ]:
if 'daily_df' in locals() and not daily_df.empty:
    fig_trend = ev.plot_trend_chart(daily_df, output_path="../outputs/charts/trend_chart.png")
    plt.show()
else:
    print("Awaiting daily dataset to plot trends.")

## 4. Anomaly Detection & Outlier Attribution

Executes `IsolationForest(contamination=0.05, random_state=42)` on daily revenue, cost, and profit. Anomalies are classified using historical baselines strictly without future leakage, and attributed to driver categories.

In [ ]:
if 'daily_df' in locals() and not daily_df.empty:
    anomaly_results = ad.detect_anomalies(daily_df, transactions_df=cleaned_txns)
    anomaly_eval = ev.evaluate_anomalies(anomaly_results)
    print(f"Flagged {anomaly_eval['flagged_days']} anomalous days ({anomaly_eval['anomaly_percentage']}% of monitored timeline).")
    print("Anomaly Types:", anomaly_eval['anomaly_type_counts'])
    
    fig_anomaly = ev.plot_anomaly_chart(anomaly_results, output_path="../outputs/charts/anomaly_chart.png")
    plt.show()
else:
    print("Awaiting daily dataset to run anomaly detection.")

## 5. Predictive Sales Forecasting

Splits data chronologically into earliest 80% train and latest 20% test without shuffling. Generates recursive walk-forward predictions and a genuine 7-day future forecast beyond the last observed date.

In [ ]:
if 'daily_df' in locals() and not daily_df.empty:
    forecast_result = fc.train_forecast_model(daily_df, config={"horizon": 7})
    print("Fitted LinearRegression model coefficients:", forecast_result.metadata['coefficients'])
    print(f"Empirical MAE uncertainty margin: +/-${forecast_result.metadata['uncertainty_margin']:.2f}")
    
    fig_forecast = ev.plot_forecast_chart(forecast_result, historical_df=daily_df, output_path="../outputs/charts/forecast_chart.png")
    plt.show()
else:
    print("Awaiting daily dataset to run sales forecasting.")

## 6. Model Evaluation & Naive Baseline Comparison

Evaluates held-out test predictions against actual revenue and benchmarks performance against a naive last-observed-value baseline.

In [ ]:
if 'forecast_result' in locals():
    forecast_eval = ev.evaluate_forecast(forecast_result)
    print("Forecast Evaluation Summary:")
    print(f" - Model MAE: ${forecast_eval['model_mae']:.2f}")
    print(f" - Model RMSE: ${forecast_eval['model_rmse']:.2f}")
    print(f" - Naive Baseline MAE: ${forecast_eval['naive_mae']:.2f}")
    print(f" - Relative Improvement: {forecast_eval['mae_improvement_pct']:+.1f}%")
    print(f" - Outperforms Naive Baseline: {forecast_eval['outperforms_baseline']}")
else:
    print("Awaiting forecast result to evaluate.")

## 7. Artifacts & Deliverables Export

Saves final CSV deliverables (`anomaly_results.csv`, `forecast_results.csv`) and generates the comprehensive markdown evaluation report (`reports/ml_evaluation_report.md`).

In [ ]:
out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

if 'anomaly_results' in locals() and 'forecast_result' in locals():
    # Export anomaly results CSV
    anomaly_csv = out_dir / "anomaly_results.csv"
    anomaly_results.to_csv(anomaly_csv, index=False)
    
    # Export forecast results CSV
    forecast_csv = out_dir / "forecast_results.csv"
    forecast_export_cols = ["forecast_date", "actual_revenue", "predicted_revenue", "lower_bound", "upper_bound"]
    if "naive_revenue" in forecast_result.forecast_df.columns:
        forecast_export_cols.append("naive_revenue")
    forecast_result.forecast_df[forecast_export_cols].to_csv(forecast_csv, index=False)
    
    # Generate evaluation report markdown
    report_md = Path("../reports/ml_evaluation_report.md")
    ev.generate_markdown_report(
        data_summary=data_summary,
        anomaly_metrics=anomaly_eval,
        forecast_metrics=forecast_eval,
        forecast_metadata=forecast_result.metadata,
        output_path=report_md,
    )
    print(f"Deliverables successfully exported to {out_dir} and {report_md}.")
else:
    print("Run preceding sections to generate deliverables.")